In [ ]:
import pandas as pd
import re
import numpy as np

# pliki wejściowe + wyjściowy do zapisu
files = ["congress_articles.csv", "congress_articles_2turn.csv"]
output_file = "congress_articles_fixed.csv"

# listy wykluczeń (wielki filtr do śmieciowych wpisów)

# 1. Słowa zakazane w subject, description i title (geografia USA, formaty, śmieci)
### lista stanów USA (małe litery dla case-insensitive)
us_states = [
    "alabama", "alaska", "arizona", "arkansas", "california", "colorado", "connecticut", "delaware", "florida", "georgia",
    "hawaii", "idaho", "illinois", "indiana", "iowa", "kansas", "kentucky", "louisiana", "maine", "maryland",
    "massachusetts", "michigan", "minnesota", "mississippi", "missouri", "montana", "nebraska", "nevada", "new hampshire",
    "new jersey", "new mexico", "new york", "north carolina", "north dakota", "ohio", "oklahoma", "oregon", "pennsylvania",
    "rhode island", "south carolina", "south dakota", "tennessee", "texas", "utah", "vermont", "virginia", "washington",
    "west virginia", "wisconsin", "wyoming"
]

other_garbage = [
    "united states", "australia", "america", "republic of chad", "republic of mozambique", 
    "kenya gazette", "republic of angola", "republic of the philippines", "venezuela", "haiti", 
    "new zealand", "atlanta", "saint louis", "oneida county", "cook islands", "aruba",
    "computer program language", "military aircraft", "aviation", "locomotive", 
    "photographic print", "portrait photograph", "interview", "photograph", "stereograph", "philadelphia"
]

global_bad_words = us_states + other_garbage

# 2. Słowa zakazane tylko w tytule (encyklopedie, sekcje biblioteczne)
title_bad_words = [
    "lcsh section", "library of congress classification", "werner encyclopædia", "americana", 
    "international encyclopaedia", "universal cyclopædia", "new imperial encyclopedia", 
    "gazette of romania", "historical reference book", "comprehensive reference book", "nasa", 
    "white pages", "telephone directory", "annual report", "census"
]

# 3. Słowa zakazane w creator
creator_bad_words = ["pdfmaker", "gpo", "government printing office"]


# funkcje pomocnicze

def clean_loc_text(text):
    """Spłaszcza tekst/wiersze i bezpiecznie usuwa tagi/nawiasy <...>"""
    if not isinstance(text, str):
        return ""
    # 1. Spłaszczanie
    text = re.sub(r'\s+', ' ', text).strip()
    # 2. Usuwanie znanych tagów HTML (z zawartością lub bez, ale tu raczej strukturalne)
    text = re.sub(r'</?(a|p|span|div|br|i|b|em|strong|h\d)[^>]*>', ' ', text, flags=re.I)
    # 3. Usuwanie samych znaków < i > (zostawiając treść pomiędzy typu np.: 'chapters I-VII')
    text = re.sub(r'[<>]', '', text)
    # 4. Usuwanie frazy o wersji cyfrowej
    text = text.replace("Also available in digital form.", "")
    return text.strip()

def extract_year(date_str):
    """Wyciąga rok (4 cyfry) z daty typu 2017-01-01"""
    if pd.isna(date_str):
        return np.nan
    match = re.search(r'\b(1\d{3}|20[0-2]\d)\b', str(date_str))
    if match:
        return int(match.group(1))
    return np.nan


In [ ]:
## przetwarzanie i czyszczenie danych

print("1. Wczytywanie i łączenie plików...")
try:
    df_list = [pd.read_csv(f, on_bad_lines='skip', dtype=str) for f in files]
    df = pd.concat(df_list, ignore_index=True)
    print(f"Połączono. Łącznie wierszy na start: {len(df):,}")
except Exception as e:
    print(f"Błąd wczytywania: {e}")
    df = pd.DataFrame()

if not df.empty:
    # wstępne czyszczenie tekstów
    print("2. Czyszczenie tekstów (tagi, entery)...")
    cols_to_clean = ['title', 'description', 'subject', 'creator', 'publisher']
    for col in cols_to_clean:
        if col in df.columns:
            df[col] = df[col].astype(str).replace('nan', '').apply(clean_loc_text)

    # duży filtr (usuwanie śmieci)
    print("3. Filtrowanie śmieci (USA, encyklopedie, gazety)...")
    initial_count = len(df)
    
    ### maski logiczne (jesli True = wiersz jest śmieciem)
    # sprawdzanie subject + description + title pod kątem globalnych śmieci
    content_to_check = df['subject'] + " " + df['description'] + " " + df['title']
    content_to_check = content_to_check.str.lower()
    
    mask_global = content_to_check.apply(lambda x: any(bad in x for bad in global_bad_words))
    
    # sprawdzanie tylko tytułu
    mask_title = df['title'].str.lower().apply(lambda x: any(bad in x for bad in title_bad_words))
    
    # sprawdzanie creator
    mask_creator = df['creator'].str.lower().apply(lambda x: any(bad in x for bad in creator_bad_words))
    
    # połączenie mask (jeśli którykolwiek warunek spełniony -> usuwanie)
    mask_trash = mask_global | mask_title | mask_creator
    
    # usuwanie śmieci
    df = df[~mask_trash]
    print(f"Usunięto {initial_count - len(df):,} rekordów uznanych za śmieci.")

    # usuwanie pustych (brak description/abstract)
    print("4. Usuwanie wierszy bez opisu (description)...")
    before_drop = len(df)
    df = df[df['description'].str.len() > 10] # musi być coś więcej niż pusty string
    print(f"Usunięto {before_drop - len(df):,} pustych wierszy.")

    # naprawa roku
    print("5. Formatowanie daty...")
    if 'date' in df.columns:
        df['year'] = df['date'].apply(extract_year).astype('Int64')

    # deduplikacja
    print("6. Deduplikacja (wg title + description)...")
    before_dedup = len(df)
    df = df.drop_duplicates(subset=['title', 'description'], keep='first')
    print(f"Usunięto {before_dedup - len(df):,} duplikatów.")

    # zapis (tylko te final_cols)
    print(f"7. Zapisywanie do {output_file}...")
    # tylko istotne kolumny - mapowanie nazw na wspólny standard np. description -> abstract
    df.rename(columns={'description': 'abstract'}, inplace=True)
    
    final_cols = ['keyword', 'title', 'creator', 'abstract', 'year', 'publisher', 'subject', 'item_url']
    
    cols_to_save = [c for c in final_cols if c in df.columns]
    
    df[cols_to_save].to_csv(output_file, index=False, encoding='utf-8')

    # podgląd
    print("\nWyniki:")
    print(df[['title', 'year']].head(5))
    print(f"Liczba końcowa rekordów: {len(df):,}")
